***
# Homework 11: MapReduce using `PySpark`

**Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** April 25th, 2024
***

## 1.) Preliminaries: Set up a Storage Bucket (2 points, spent $\approx$ 5 minutes)

**Before we get started, create a storage bucket for this project, just like you did for Homework 10, this time called `NetID-stat606s24-hw11`, where `NetID` is your Wisconsin `NetID` in all lower-case letters. All settings should be the same as the bucket you created for Homework 10.**

Completed the above procedure.

## 2.) Warmup: Interactive PySpark on GCP (3 points, spent $\approx$ 25 minutes)

**Before we can do anything in `PySpark`, we have to get a server up and running. Sign in to Google Cloud Platform, and make sure that you are in your project that you created in Homework 10. Recall that this project should be named `NetID-stat606s24`, where `NetID` is your Wisconsin `NetID` in all lower-case letters. Open Google Cloud Console and type:**

<h5 align="center"> gcloud dataproc clusters create CLUSTERNAME --region=REGION </h5>

**where `CLUSTERNAME` is the name you wish to give your cluster (e.g., stat606hw11 or something like that; you are free to name this however you like) and `REGION` is a valid region. You will need to wait a few minutes while Google Cloud sets up your cluster (i.e., gets some computers to serve as your nodes, installs necessary software on those computers, etc.). Once this process finishes, you will see a message to the effect of `Created [CLUSTERNAME] Cluster placed in zone [REGION]`. Once you have created this cluster, you should see it listed when you call**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**in the console, where `REGION` is the same as the argument supplied when you created the cluster.**

***Important warning:* any time you finish a working session (e.g., to take a break and come back again later), consider deleting your cluster with:**

<h5 align="center"> gcloud dataproc clusters delete CLUSTERNAME --region=REGION </h5>

**to ensure that you are not paying to leave a cluster sitting unused. Of course, when you come back to continue working, you will have to spin up the cluster again by following the instructions above. Bear in mind that any files that you create on the cluster are lost when you delete it, so be sure to move any files you want to keep into a storage bucket (we discuss this point at more length below).**

Completed the above procedure.

Given below is the terminal chunk that created the requested cluster:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters create ssudhir2-stat606s24-hw11 --region=us-east1
Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/30ba5125-82fb-30a3-99ef-e0aa2c4966c1].
Waiting for cluster creation operation...                                        
WARNING: No image specified. Using the default image version. It is recommended to select a specific image version in production, as the default image version may change at any time.
WARNING: Failed to validate permissions required for default service account: '89254757867-compute@developer.gserviceaccount.com'. Cluster creation could still be successful if required permissions have been granted to the respective service accounts as mentioned in the document https://cloud.google.com/dataproc/docs/concepts/configuring-clusters/service-accounts#dataproc_service_accounts_2. This could be due to Cloud Resource Manager API hasn't been enabled in your project '89254757867' before or it is disabled. Enable it by visiting 'https://console.developers.google.com/apis/api/cloudresourcemanager.googleapis.com/overview?project=89254757867'.
WARNING: The firewall rules for specified network or subnetwork would allow ingress traffic from 0.0.0.0/0, which could be a security risk.
WARNING: Unable to validate the staging bucket lifecycle configuration of the bucket 'dataproc-staging-us-east1-89254757867-y0f1tueq' due to an internal error, Please make sure that the provided bucket doesn't have any delete rules set.
Waiting for cluster creation operation...done.                                   
Created [https://dataproc.googleapis.com/v1/projects/ssudhir2-stat606s24/regions/us-east1/clusters/ssudhir2-stat606s24-hw11] Cluster placed in zone [us-east1-c].
```

**Okay, now that we have a cluster up and running, let’s try running an interactive `PySpark` session. To do that, we need to log onto our cluster. We will `ssh` to the master node on your Dataproc cluster. Double-check that your Dataproc cluster is up and running by calling**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**again (`REGION` should be set to whatever region you requested when you created the cluster).** 

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-c
SCHEDULED_DELETE:
```

**If a cluster shows up in the list, go to the VM Instances dashboard, where you should see a few entries listed. These correspond to the nodes in your cluster. The names of these instances should all be prefixed with your cluster name. One of them should end with `-m`. This is the master node in your cluster. To `ssh` to it (i.e., log on to that machine), type the command**

<h5 align="center"> gcloud compute ssh MASTERNODE --project=PROJECT --zone=ZONE </h5>

**in the console, where `MASTERNODE` is the name of your cluster with the added suffix `-m` (something like `CLUSTERNAME-m`), `PROJECT` is the name of your project (something like `NetID-stat606s24`), and `ZONE` is the specific zone that your cluster is in. This will have a form like `REGION` or `REGION-X`, where `REGION` is your specific region specified when you launched the cluster, and `X` is a letter or number. If you’re not sure, you can find the zone of your cluster in the “Zone” column of the VM instances dashboard.**

**Note: you may be prompted to create an RSA key pair when logging on to your master node. Go ahead and create a password for this if you wish, or feel free to use the “no password” option, since we won’t be working with any sensitive files in this exercise. Of course, when working in an actual production environment, you should be careful to follow good security practices such as using secure passwords, encryption, etc.**

**If all goes well, it won’t look like much has changed, except you’ll see that your prompt in the console has changed to something like `NetID@CLUSTERNAME-m`. Alternatively, you can type the command hostname in the console, which should produce an output of the form `CLUSTERNAME-m`.**

Currently at the SSH-in-browser bash shell terminal:

```bash
Linux ssudhir2-stat606s24-hw11-m 5.10.0-0.deb10.16-cloud-amd64 #1 SMP Debian 5.10.127-2~bpo10+1 (2022-07-28) x86_64

The programs included with the Debian GNU/Linux system are free software;
the exact distribution terms for each program are described in the
individual files in /usr/share/doc/*/copyright.

Debian GNU/Linux comes with ABSOLUTELY NO WARRANTY, to the extent
permitted by applicable law.
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ 
```

**Now you can start an interactive PySpark session by typing `pyspark` in the console.**


**When you do this, you’ll see some text appear, giving some setup information and information about the version of Spark, and then you’ll see the interactive prompt (>>>).** 

**The `numbers.txt` file from lecture is available at:**

<h5 align="center"> gs://uw-stat606s24-hw11/numbers.txt </h5>

**Read it into an RDD in your PySpark interactive session and use a sequence of RDD transformations and RDD actions to compute how many of the numbers in the file are prime.** 

**You may make use of the function is_prime, which is defined in the Python file:**

<h5 align="center"> gs://uw-stat606s24-hw11/prime.py </h5>

**Save the answer in a variable called number_of_primes in your Jupyter notebook file for submission. Note: you can quit an interactive PySpark session either by typing `quit()` at the prompt or by typing ctrl-D.3**

**Please also copy-paste into your Jupyter notebook file the sequence of PySpark commands that you ran to obtain this answer. Important: paste these into a Raw NBConvert or Markdown cell, not a Code cell. If you paste these commands into a Jupyter Code cell, the grader script will try to run your PySpark commands in plain old Python, which will cause errors.**

**Reminder: if you aren’t going to continue working on the next problem immediately, save GCP credits by deleting your cluster.**

First, we go to the `CLUSTERNAME-m` bash command terminal to check all files that are located in `gs://uw-stat606s24-hw11`:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://uw-stat606s24-hw11/
gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv
gs://uw-stat606s24-hw11/numbers.txt
gs://uw-stat606s24-hw11/prime.py
gs://uw-stat606s24-hw11/ps_wordcount.py
gs://uw-stat606s24-hw11/war_and_peace.txt
```

We can see that both our `numbers.txt` as well as `prime.py` are located in the cloud directory.

We now use `cat` to check the contents of the file as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://uw-stat606s24-hw11/prime.py
import math

def is_prime(n):
    if n <= 1: # Primes must be naturals; 1 is not prime.
        return False
    for x in range(2,int(math.sqrt(n))+1):
        if n%x==0:
            return False
    return True 
```

Now, we load into Pyspark as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ pyspark
Python 3.8.15 | packaged by conda-forge | (default, Nov 22 2022, 08:46:39) 
[GCC 10.4.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/28 04:43:26 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/28 04:43:27 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.1.3
      /_/

Using Python version 3.8.15 (default, Nov 22 2022 08:46:39)
Spark context Web UI available at http://ssudhir2-stat606s24-hw11-m.us-east1-c.c.ssudhir2-stat606s24.internal:39851
Spark context available as 'sc' (master = yarn, app id = application_1714278108368_0002).
SparkSession available as 'spark'.
>>>
```

We first load the `numbers.txt` data into pyspark using `sc.textfile()` and `collect()`, then we convert the list of string into a list of integers as follows:

```bash
>>> data = sc.textFile('gs://uw-stat606s24-hw11/numbers.txt')
>>> data.collect()
[Stage 0:>                                                          (0 + 2) /[Stage 0:=============================>                             (1 + 1) /                                                                             ['10', '23', '16', '7', '12', '0', '1', '1', '2', '3', '5', '8', '-1', '42', '64', '101', '-101', '3']
>>> numbers = data.collect()
```

We now load the `prime.py` file into pyspark using `sc.addPyfile()` and the relevant import statements as follows:

```bash
>>> sc.addPyFile('gs://uw-stat606s24-hw11/prime.py')
>>> from prime import *
```

Now, we run the `is_prime()` function from `prime.py` on the list `numbers`, and add up all `True` values to get our answer:

```bash
>>> prime_results = [is_prime(int(num)) for num in numbers]
>>> prime_results
[False, True, False, True, False, False, False, False, True, True, True, False, False, False, False, True, False, True]
>>> number_of_primes = sum(prime_results)
>>> print(number_of_primes)
7
>>> quit()
ssudhir2@ssudhir2-stat606s24-hw11-m:~$
```

In [1]:
number_of_primes = 7

## 3.) Submitting a Job to Spark (6 points, spent $\approx$ 15 minutes)

**Now let’s try writing a PySpark script and submitting it to your Dataproc server.**

**First things first: make sure that you have a Dataproc cluster up and running by typing**

<h5 align="center"> gcloud dataproc clusters list --region=REGION </h5>

**where `REGION` is the region you specified upon cluster creation. Alternatively, you can pull up the VM instances dashboard to see a list of your currently-running VM instances (this list will include any running Dataproc clusters).** 

**If you don’t have a Dataproc cluster up and running, follow the instructions from the previous problem to create one.**

Checking for clusters:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-c
SCHEDULED_DELETE: 
```

**Now let’s try running our example from lecture. The `ps_wordcount.py` script from the lecture slides is available at:**

<h5 align="center"> gs://uw-stat606s24-hw11/ps_wordcount.py </h5>

**(alternatively, you can download the demo code from this week’s lecture and upload a copy to your own storage bucket).** 

**The file at**

<h5 align="center"> gs://uw-stat606s24-hw11/war_and_peace.txt </h5>

**contains a slightly modified version of the Project Gutenberg UTF-8 copy of *Leo Tolstoy’s War and Peace* Submit a PySpark job to your Dataproc server that runs `ps_wordcount.py` on `war_and_peace.txt` and outputs the results to a directory:**

<h5 align="center"> gs://NetID-stat606s24-hw11/WP_wordcount </h5>

**where once again `NetID` is your `NetID` in all lower-case. Please also copy-paste the command that you called to launch this job into a Raw NBConvert cell or a Markdown cell in your Jupyter notebook file.**

We first use `cat` to see what the `ps_wordcount.py` file looks like:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://uw-stat606s24-hw11/ps_wordcount.py
from pyspark import SparkConf, SparkContext
import sys

# This script takes two arguments, an input file and output directory.
if len(sys.argv) != 3:
    print('Usage: ' + sys.argv[0] + ' <in> <out>')
    sys.exit(1)
inputlocation = sys.argv[1]
outputlocation = sys.argv[2]

# Set up the configuration and job context
conf = SparkConf().setAppName('WordCount')
sc = SparkContext(conf=conf)

# Read in the dataset and immediately transform all the lines into arrays.
data = sc.textFile(inputlocation)
data_flat = data.flatMap(lambda line: line.split())
wordkeys = data_flat.map(lambda w: (w.lower(),1) )
wordcounts = wordkeys.reduceByKey(lambda x,y: x+y)

# Save the results in the specified output directory.
wordcounts.saveAsTextFile(outputlocation)
sc.stop() # Let Spark know that the job is done.
```

We can see that the script takes two command line arguments, input file and output directory.

Thus, with this knowledge, we now run `ps_wordcount.py` on the bash command terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://uw-stat606s24-hw11/ps_wordcount.py gs://uw-stat606s24-hw11/war_and_peace.txt gs://ssudhir2-stat606s24-hw11/WP_wordcount
24/04/30 19:48:04 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/30 19:48:04 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/30 19:48:04 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/30 19:48:04 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/30 19:48:04 INFO org.sparkproject.jetty.util.log: Logging initialized @5330ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/30 19:48:05 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_412-b08
24/04/30 19:48:05 INFO org.sparkproject.jetty.server.Server: Started @5430ms
24/04/30 19:48:05 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@68a55f8a{HTTP/1.1, (http/1.1)}{0.0.0.0:40749}
24/04/30 19:48:05 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8032
24/04/30 19:48:06 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.19:10200
24/04/30 19:48:06 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/30 19:48:06 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/30 19:48:07 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714505761919_0002
24/04/30 19:48:08 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8030
24/04/30 19:48:09 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_get_file_status. latencyMs=331; previousMaxLatencyMs=204; operationCount=2; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 19:48:09 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/30 19:48:09 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=334; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 19:48:10 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_create. latencyMs=487; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0002.inprogress
24/04/30 19:48:11 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/30 19:48:21 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_wordcount/' directory.
24/04/30 19:48:21 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=499; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_wordcount/_temporary
24/04/30 19:48:21 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=196; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_wordcount/_SUCCESS
24/04/30 19:48:21 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@68a55f8a{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/30 19:48:22 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=414; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0002.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0002)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/WP_wordcount/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-00000
gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-00001
```

**Concatenate the output of your script and store it in a file in your storage bucket at:**

<h5 align="center"> gs://NetID-stat606s24-hw11/wp_output.txt. </h5>

**please also include a copy of this file in your submission.**

Concatenating `part-00000` and `part-00001` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_wordcount/part-* > wp_output.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
wp_output.txt
```

Now, we check if our output looks as we expect it to look:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ sed -n -e 1,15p -e 10p wp_output.txt
('peace', 61)
('leo', 1)
('book', 51)
('one:', 3)
('1805', 6)
('chapter', 366)
('i', 3209)
('“well,', 264)
('genoa', 3)
('lucca', 2)
('lucca', 2)
('are', 1163)
('now', 956)
('just', 535)
('family', 80)
('of', 14787)
```

Now, we transfer `wp_output.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp wp_output.txt gs://ssudhir2-stat606s24-hw11/
Copying file://wp_output.txt [Content-Type=text/plain]...
/ [1 files][648.9 KiB/648.9 KiB]                                                
Operation completed over 1 objects/648.9 KiB.                               
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

## 4.) Climate Data Revisited (9 points, spent $\approx$ 50 minutes)

**I used NOAA’s Climate Data Online service to collect daily historical temperature data for Madison, WI, which has been gathered at Dane County Airport since 1939.** 

**I have made this data available on GCP at:**

<h5 align="center"> gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv </h5>

**Each line of this file has the form: `DATE,TMAX,TMIN`**

**where `TMAX` and `TMIN` are integers describing the maximum and minimum temperatures (in degrees Fahrenheit) on a given day, and `DATE` encodes a date in the form `YYYYMMDD`.**

**You can see a few lines of the file by writing something like**

<h5 align="center"> gsutil cat -r 0-89 gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv </h5>

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat -r 0-89 gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv
19391001,67,33
19391002,70,38
19391003,74,48
19391004,81,51
19391005,70,56
19391006,77,45
```

**to print out the first bytes of the file (six lines, at 15 bytes per line including the trailing new lines). This `-r` flag to the `gsutil cat` command is the closest thing (to the best of my knowledge, anyway) that `gsutil` has to the UNIX `head` command.** 

**Important: be careful when performing read operations like this with very large files. Reading multiple GBs or, worse, TBs of text into less or a similar command-line program can be very slow!**

**Write a PySpark script that reads two arguments from the command line, corresponding to an input file and an output directory, in that order (the same as the arguments for `ps_wordcount.py`) and computes the average maximum and minimum temperature for every year in the data set. The output should be of the form: `YYYY, avgmax, avgmin`**

**where `YYYY` is an integer encoding a year, and `avgmax` and `avgmin` are floats encoding the average maximum and minimum temperatures, respectively, for that year.**

**Note: the precise formatting here does not matter—just make sure that your output has a line for each year in the data set and the maximum and minimum temperature are ordered correctly. So, for example, an output like `YYYY, (avgmax, avgmin)` is also fine.** 

**Save your script in a file called `ps_year_avgs.py` and include copies in both your storage bucket and your submission. If you wrote any additional Python code (e.g., function definitions in a separate Python file), please also include this in your submission.** 

**Hint: you may find the `reduceByKey` and `mapValues` transformations to be especially useful**

I wrote the python scripts for both `ps_year_avgs.py` and `ps_year_extremes.py` in the terminal itself by installing `emacs`. The note for this can be found at the end of the notebook.

Showing contents of `ps_year_avgs.py` file:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ cat ps_year_avgs.py

from pyspark import SparkConf, SparkContext
import sys

# this script takes two arguements, an input file, and an output directory                                                                                      
if len(sys.argv) != 3:
    print('Usage: ' + sys.argv[0] + ' <in> <out>')
    sys.exit(1)
inputlocation = sys.argv[1]
outputlocation = sys.argv[2]

# Set up the configuration and job context                                                                                                                      
conf = SparkConf().setAppName('YearlyAverages')
sc = SparkContext(conf = conf)

def change_date(line):

    '''                                                                                                                                                         
    helper function to extract relevant information from the input data                                                                                         
                                                                                                                                                                
    ------------ inputs ------------                                                                                                                            
    line :  str                                                                                                                                                 
            strings in the format 'YYYYMMDD,TMAX,TMIN'                                                                                                          
                                                                                                                                                                
    ------------ outputs ------------                                                                                                                           
    result :    tuple                                                                                                                                           
                tuples in the format (YYYY, tmax, tmin)                                                                                                         
    '''

    string_line= line.split(',')

    try:
        year = int(string_line[0][0:4])
        tmax = int(string_line[1])
        tmin = int(string_line[2])
    except:
        return None
    
    result = (year, tmax, tmin)
    return result

def get_avg(line):

    '''                                                                                                                                                         
    helper function to compute the average of the tmax and tmin for each year                                                                                   
                                                                                                                                                                
    ------------ inputs ------------                                                                                                                            
    line :  str                                                                                                                                                 
            strings in the format 'YYYYMMDD,TMAX,TMIN'                                                                                                          
                                                                                                                                                                
    ------------ outputs ------------                                                                                                                           
    result :    tuple                                                                                                                                           
                tuples in the format (YYYY, avgmax, avgmin)                                                                                                     
    '''

    year = line[0]
    values = list(line[1])
    avgmax = sum(x[1] for x in values) / len(values)
    avgmin = sum(x[2] for x in values) / len(values)
    result = (year, round(avgmax, 2), round(avgmin, 2))
    return result

# Read in the data from the input file                                                                                                                          
rdd = sc.textFile(inputlocation)              # Read data from input file                                                                                       
rdd = rdd.map(lambda line: change_date(line)) # 'YYYYMMDD,TMAX,TMIN' -> [YYYY, MM, DD, tmax, tmin]                                                              
rdd = rdd.filter(lambda x: x != None)         # Remove empty lines                                                                                              
rdd = rdd.groupBy(lambda x: x[0])             # Group by year                                                                                                   
rdd = rdd.map(lambda line: get_avg(line))     # Compute the average of the tmax and tmin for each year                                                          

# Let spark know the job is done, processes can be ended                                                                                                        
rdd.saveAsTextFile(outputlocation)
sc.stop()
```

Showing contents of `ps_year_extremes.py` file:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ cat ps_year_extremes.py

from pyspark import SparkConf, SparkContext
import sys

# this script takes two arguements, an input file, and an output directory                                                                                     
if len(sys.argv) != 3:
    print('Usage: ' + sys.argv[0] + ' <in> <out>')
    sys.exit(1)
inputlocation = sys.argv[1]
outputlocation = sys.argv[2]

# Set up the configuration and job context                                                                                                                     
conf = SparkConf().setAppName('YearlyAverages')
sc = SparkContext(conf = conf)

def change_date(line):

    '''                                                                                                                                                         
    helper function to extract relevant information from the input data                                                                                         
                                                                                                                                                                
    ------------ inputs ------------                                                                                                                            
    line :  str                                                                                                                                                 
            strings in the format 'YYYYMMDD,TMAX,TMIN'                                                                                                          
                                                                                                                                                                
    ------------ outputs ------------                                                                                                                           
    result :    tuple                                                                                                                                           
                tuples in the format (YYYY, MM-DD, tmax, tmin)                                                                                                  
    '''

    string_line = line.split(',')

    try:
        year = int(string_line[0][0:4])
        month = int(string_line[0][4:6])
        day = int(string_line[0][6:])
        tmax = int(string_line[1])
        tmin = int(string_line[2])
    except:
        return None
    
    result = (year, f'{month}-{day}', tmax, tmin)
    return result

# Read in the data from the input file                                                                                                                          
rdd = sc.textFile(inputlocation)
rdd = rdd.map(lambda line: change_date(line)) # 'YYYYMMDD,TMAX,TMIN' -> [YYYY, MM, DD, tmax, tmin]                                                              
rdd = rdd.filter(lambda x: x != None)         # Remove empty lines                                                                                              
rdd = rdd.groupBy(lambda x: x[0])             # Group by year                                                                                                   
rdd = rdd.map(  lambda x: (x[0],                           # Year                                                                                               
                sorted(x[1], key = lambda y: -y[2])[0][1], # Date with max value in the third column                                                            
                sorted(x[1], key = lambda y: y[3])[0][1],  # Date with min value in the fourth column                                                           
            ))

# Let spark know the job is done, processes can be ended                                                                                                        
rdd.saveAsTextFile(outputlocation)
sc.stop()
```

Showing contents of dataproc cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
ps_year_avgs.py  ps_year_extremes.py  wp_output.txt
```

Transferring both files to the cloud bucker:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp ps_year_avgs.py ps_year_extremes.py gs://ssudhir2-stat606s24-hw11/
Copying file://ps_year_avgs.py [Content-Type=text/x-python]...
/ [0 files][    0.0 B/  5.4 KiB]                                             / [1 files][  5.4 KiB/  5.4 KiB]                                             Copying file://ps_year_extremes.py [Content-Type=text/x-python]...
/ [1 files][  5.4 KiB/  9.3 KiB]                                             / [2 files][  9.3 KiB/  9.3 KiB]                                                
Operation completed over 2 objects/9.3 KiB.   
```

**Run `ps_year_avgs.py` on the file `NOAA_MSN_temps.csv` in PySpark on a GCP Dataproc server.** 

**Concatenate the output of your job into a single file called `avgs.txt` and save this file in your storage bucket for this homework, and please also include a copy in your submission.**

First, we check the contents of our cloud bucket `ssudhir2-stat606s24-hw11`:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

We can see that our python script `ps_year_avgs.py` and `ps_year_extremes.py` exists in our cloud bucket.

Now, we can run the script using `spark-submit` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv gs://ssudhir2-stat606s24-hw11/WP_averages/
24/04/30 20:08:32 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/30 20:08:32 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/30 20:08:32 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/30 20:08:33 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/30 20:08:33 INFO org.sparkproject.jetty.util.log: Logging initialized @5578ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/30 20:08:33 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_412-b08
24/04/30 20:08:33 INFO org.sparkproject.jetty.server.Server: Started @5679ms
24/04/30 20:08:33 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@203d28a8{HTTP/1.1, (http/1.1)}{0.0.0.0:40851}
24/04/30 20:08:33 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8032
24/04/30 20:08:34 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.19:10200
24/04/30 20:08:34 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/30 20:08:34 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/30 20:08:35 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714505761919_0004
24/04/30 20:08:36 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8030
24/04/30 20:08:37 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_get_file_status. latencyMs=322; previousMaxLatencyMs=210; operationCount=2; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 20:08:38 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/30 20:08:38 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=373; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 20:08:38 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_create. latencyMs=520; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0004.inprogress
24/04/30 20:08:39 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/30 20:08:48 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_averages/' directory.
24/04/30 20:08:48 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=449; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_averages/_temporary
24/04/30 20:08:48 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=179; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_averages/_SUCCESS
24/04/30 20:08:48 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@203d28a8{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/30 20:08:49 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=418; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0004.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0004)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_averages/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00000
gs://ssudhir2-stat606s24-hw11/WP_averages/part-00001
```

Concatenating `part-00000` and `part-00001` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_averages/part-* | sort -n -k1 > avgs.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
avgs.txt  ps_year_avgs.py  ps_year_extremes.py  wp_output.txt
```

Now,  we check the contents to see whether `avgs.txt` looks as we expect it to look:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ cat avgs.txt
(1939, 49.5, 30.16)
(1940, 55.29, 35.58)
(1941, 56.84, 40.2)
(1942, 54.91, 38.07)
(1943, 54.59, 36.3)
(1944, 55.79, 38.68)
(1945, 54.3, 37.45)
(1946, 57.36, 38.67)
(1947, 54.41, 38.36)
(1948, 57.81, 35.66)
(1949, 58.67, 36.65)
(1950, 54.87, 33.93)
(1951, 54.11, 33.86)
(1952, 57.88, 36.96)
(1953, 59.46, 38.24)
(1954, 58.34, 37.55)
(1955, 58.47, 36.55)
(1956, 57.86, 36.14)
(1957, 56.82, 36.15)
(1958, 57.25, 34.45)
(1959, 56.25, 35.43)
(1960, 54.89, 34.37)
(1961, 56.08, 35.04)
(1962, 54.8, 34.46)
(1963, 56.17, 33.37)
(1964, 58.48, 35.41)
(1965, 55.32, 34.11)
(1966, 55.7, 32.32)
(1967, 54.78, 32.98)
(1968, 56.98, 35.27)
(1969, 54.46, 33.93)
(1970, 56.05, 34.63)
(1971, 57.49, 34.38)
(1972, 53.65, 32.42)
(1973, 57.68, 37.77)
(1974, 56.44, 35.22)
(1975, 57.88, 36.05)
(1976, 57.72, 31.89)
(1977, 56.79, 34.53)
(1978, 55.03, 32.89)
(1979, 54.76, 33.0)
(1980, 55.4, 33.89)
(1981, 57.63, 35.47)
(1982, 54.79, 33.74)
(1983, 55.4, 35.65)
(1984, 56.06, 35.8)
(1985, 54.97, 34.44)
(1986, 55.98, 36.62)
(1987, 58.99, 38.93)
(1988, 58.29, 35.2)
(1989, 55.64, 34.12)
(1990, 58.5, 37.78)
(1991, 58.18, 37.37)
(1992, 56.02, 36.85)
(1993, 55.73, 36.92)
(1994, 57.76, 35.95)
(1995, 57.21, 36.98)
(1996, 53.36, 33.99)
(1997, 54.36, 36.18)
(1998, 58.89, 40.93)
(1999, 58.16, 38.08)
(2000, 56.24, 37.11)
(2001, 57.46, 38.78)
(2002, 57.3, 38.18)
(2003, 56.38, 35.79)
(2004, 56.52, 37.57)
(2005, 58.18, 38.5)
(2006, 57.93, 39.36)
(2007, 57.35, 38.49)
(2008, 54.78, 35.54)
(2009, 55.23, 36.0)
(2010, 58.13, 38.69)
(2011, 57.29, 38.34)
(2012, 61.99, 40.75)
(2013, 55.32, 36.29)
(2014, 54.45, 35.24)
(2015, 57.86, 38.75)
(2016, 58.34, 40.35)
(2017, 57.64, 39.07)
(2018, 56.01, 37.84)
(2019, 55.26, 37.27)
(2020, 57.52, 38.82)
(2021, 58.87, 38.99)
(2022, 56.65, 37.6)
(2023, 59.61, 40.36)
```

Now, we transfer `avgs.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp avgs.txt gs://ssudhir2-stat606s24-hw11/
Copying file://avgs.txt [Content-Type=text/plain]...
/ [0 files][    0.0 B/  1.7 KiB]                                             / [1 files][  1.7 KiB/  1.7 KiB]                                                
Operation completed over 1 objects/1.7 KiB.                      
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/avgs.txt
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

**Write a PySpark script whose command line arguments are the same as those of `ps_wordcount.py` and `ps_year_avgs.py` and that computes, for each year in the data set, the day on which the maximum temperature was achieved and the day on which the minimum temperature was achieved (you may break ties as you see fit).**

**That is, each row of the output should be of a form like: `YYYY, MM-DD, mm-dd` where `MM-DD` encodes the month and day on which the maximum occurred and `mm-dd` encodes the month and day on which the minimum occurred.** 

**Note: the precise formatting here does not matter—just make sure that your output has a line for each year in the data set and the maximum and minimum temperature days are ordered correctly and in the correct `MM-DD` format, that is fine. So, for example, an output like: `YYYY, (MM-DD, mm-dd)` or `YYYY, 'MM-DD' 'mm-dd'` is also fine.** 

**Save your script in a file called `ps_year_extremes.py`. Please include a copy of this script in your storage bucket and include a copy in your submission.**

**If you wrote any additional Python code (e.g., function definitions in a separate Python file), please also include this in your submission.** 

**Hint: you may find it easiest to find the maximum and minimum separately, and then combine the two RDDs using the RDD transformation join.**

Completed the above procedure in part (a.) itself. Check above for further details.

**Run your `ps_year_extremes.py` script on the file `NOAA_MSN_temps.csv` in PySpark on a GCP Dataproc server.** 

**Concatenate the output of your job into a single file called `extremes.txt` and save this file in your storage bucket for this homework, and please also include a copy in your submission.**

ALl the aforementioned steps have already been completed, we can run the script using `spark-submit` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ spark-submit gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py gs://uw-stat606s24-hw11/NOAA_MSN_temps.csv gs://ssudhir2-stat606s24-hw11/WP_extremes/
24/04/30 20:13:07 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/30 20:13:07 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/30 20:13:07 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/30 20:13:07 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
24/04/30 20:13:08 INFO org.sparkproject.jetty.util.log: Logging initialized @5366ms to org.sparkproject.jetty.util.log.Slf4jLog
24/04/30 20:13:08 INFO org.sparkproject.jetty.server.Server: jetty-9.4.40.v20210413; built: 2021-04-13T20:42:42.668Z; git: b881a572662e1943a14ae12e7e1207989f218b74; jvm 1.8.0_412-b08
24/04/30 20:13:08 INFO org.sparkproject.jetty.server.Server: Started @5486ms
24/04/30 20:13:08 INFO org.sparkproject.jetty.server.AbstractConnector: Started ServerConnector@3b14e2b3{HTTP/1.1, (http/1.1)}{0.0.0.0:44515}
24/04/30 20:13:08 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8032
24/04/30 20:13:09 INFO org.apache.hadoop.yarn.client.AHSProxy: Connecting to Application History server at ssudhir2-stat606s24-hw11-m/10.142.0.19:10200
24/04/30 20:13:09 INFO org.apache.hadoop.conf.Configuration: resource-types.xml not found
24/04/30 20:13:09 INFO org.apache.hadoop.yarn.util.resource.ResourceUtils: Unable to find 'resource-types.xml'.
24/04/30 20:13:10 INFO org.apache.hadoop.yarn.client.api.impl.YarnClientImpl: Submitted application application_1714505761919_0005
24/04/30 20:13:11 INFO org.apache.hadoop.yarn.client.RMProxy: Connecting to ResourceManager at ssudhir2-stat606s24-hw11-m/10.142.0.19:8030
24/04/30 20:13:12 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_get_file_status. latencyMs=285; previousMaxLatencyMs=211; operationCount=2; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 20:13:13 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl: Ignoring exception of type GoogleJsonResponseException; verified object already exists with desired state.
24/04/30 20:13:13 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_mkdirs. latencyMs=381; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history
24/04/30 20:13:13 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_create. latencyMs=419; previousMaxLatencyMs=0; operationCount=1; context=gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0005.inprogress
24/04/30 20:13:14 WARN org.apache.hadoop.util.concurrent.ExecutorHelper: Thread (Thread[GetFileInfo #1,5,main]) interrupted: 
java.lang.InterruptedException
        at com.google.common.util.concurrent.AbstractFuture.get(AbstractFuture.java:510)
        at com.google.common.util.concurrent.FluentFuture$TrustedFuture.get(FluentFuture.java:88)
        at org.apache.hadoop.util.concurrent.ExecutorHelper.logThrowableFromAfterExecute(ExecutorHelper.java:48)
        at org.apache.hadoop.util.concurrent.HadoopThreadPoolExecutor.afterExecute(HadoopThreadPoolExecutor.java:90)
        at java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1157)
        at java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:624)
        at java.lang.Thread.run(Thread.java:750)
24/04/30 20:13:14 INFO org.apache.hadoop.mapred.FileInputFormat: Total input files to process : 1
24/04/30 20:13:23 INFO com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystem: Successfully repaired 'gs://ssudhir2-stat606s24-hw11/WP_extremes/' directory.
24/04/30 20:13:23 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_delete. latencyMs=432; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_extremes/_temporary
24/04/30 20:13:23 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation stream_write_close_operations. latencyMs=176; previousMaxLatencyMs=0; operationCount=1; context=gs://ssudhir2-stat606s24-hw11/WP_extremes/_SUCCESS
24/04/30 20:13:23 INFO org.sparkproject.jetty.server.AbstractConnector: Stopped Spark@3b14e2b3{HTTP/1.1, (http/1.1)}{0.0.0.0:0}
24/04/30 20:13:24 INFO com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics: Detected potential high latency for operation op_rename. latencyMs=391; previousMaxLatencyMs=0; operationCount=1; context=rename(gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0005.inprogress -> gs://dataproc-temp-us-east1-89254757867-nbcdhrwu/dc22487d-badf-41bc-a27c-e6ff40f6b563/spark-job-history/application_1714505761919_0005)
```

Now, we check if the job was run successfully on our created cluster:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls  gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_extremes/_SUCCESS
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00000
gs://ssudhir2-stat606s24-hw11/WP_extremes/part-00001
```

Concatenating `part-00000` and `part-00001` in our bash terminal as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cat gs://ssudhir2-stat606s24-hw11/WP_extremes/part-* | sort -n -k1 > extremes.txt
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ ls
avgs.txt  extremes.txt  ps_year_avgs.py  ps_year_extremes.py  wp_output.txt
```

Now, we check whether `extremes.txt` looks like how we expect it to look:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ cat extremes.txt
(1939, '10-7', '12-31')
(1940, '7-24', '1-18')
(1941, '7-29', '2-19')
(1942, '7-17', '1-8')
(1943, '7-13', '1-20')
(1944, '8-10', '12-24')
(1945, '7-24', '12-18')
(1946, '8-16', '1-26')
(1947, '8-4', '2-4')
(1948, '8-24', '1-18')
(1949, '7-4', '1-20')
(1950, '5-24', '12-27')
(1951, '7-24', '1-30')
(1952, '6-28', '1-24')
(1953, '8-31', '1-6')
(1954, '6-25', '1-17')
(1955, '8-21', '1-27')
(1956, '6-10', '1-21')
(1957, '7-20', '1-14')
(1958, '8-3', '12-10')
(1959, '8-21', '2-2')
(1960, '9-7', '3-6')
(1961, '8-30', '12-15')
(1962, '8-23', '3-1')
(1963, '7-1', '1-15')
(1964, '7-19', '12-17')
(1965, '7-22', '1-29')
(1966, '7-10', '1-29')
(1967, '5-26', '1-18')
(1968, '8-6', '1-7')
(1969, '8-30', '1-12')
(1970, '6-30', '1-19')
(1971, '6-27', '1-18')
(1972, '7-11', '1-15')
(1973, '7-8', '2-17')
(1974, '7-13', '1-12')
(1975, '5-19', '2-9')
(1976, '7-10', '2-2')
(1977, '7-14', '1-9')
(1978, '9-8', '1-30')
(1979, '8-7', '1-11')
(1980, '7-7', '2-4')
(1981, '7-12', '2-11')
(1982, '8-3', '1-17')
(1983, '8-19', '12-19')
(1984, '8-29', '1-21')
(1985, '9-7', '2-1')
(1986, '7-18', '1-8')
(1987, '6-14', '1-24')
(1988, '8-16', '1-7')
(1989, '7-9', '12-21')
(1990, '7-4', '12-23')
(1991, '6-29', '1-30')
(1992, '8-9', '1-15')
(1993, '8-11', '1-18')
(1994, '6-18', '1-18')
(1995, '7-13', '1-8')
(1996, '6-29', '2-3')
(1997, '7-26', '1-17')
(1998, '9-13', '1-13')
(1999, '7-30', '1-5')
(2000, '9-1', '12-25')
(2001, '7-31', '1-2')
(2002, '7-21', '3-4')
(2003, '8-20', '1-23')
(2004, '7-13', '1-30')
(2005, '7-16', '1-23')
(2006, '7-31', '2-18')
(2007, '7-31', '2-5')
(2008, '9-2', '2-21')
(2009, '6-23', '1-16')
(2010, '8-12', '1-9')
(2011, '7-20', '1-23')
(2012, '7-5', '1-19')
(2013, '9-10', '12-30')
(2014, '7-22', '1-6')
(2015, '8-14', '2-23')
(2016, '7-21', '1-18')
(2017, '9-23', '12-31')
(2018, '5-27', '1-1')
(2019, '7-19', '1-30')
(2020, '7-26', '2-14')
(2021, '6-11', '2-9')
(2022, '6-14', '1-26')
(2023, '8-23', '1-31')
```

Now, we transfer `extremes.txt` to the cloud bucket `gs://ssudhir2-stat606s24-hw11/` as follows:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil cp extremes.txt gs://ssudhir2-stat606s24-hw11/
Copying file://extremes.txt [Content-Type=text/plain]...
/ [0 files][    0.0 B/  1.9 KiB]                                             / [1 files][  1.9 KiB/  1.9 KiB]                                                
Operation completed over 1 objects/1.9 KiB.                                  
```

Now, we check if our output is in the cloud directory as expected:

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ gsutil ls gs://ssudhir2-stat606s24-hw11/
gs://ssudhir2-stat606s24-hw11/avgs.txt
gs://ssudhir2-stat606s24-hw11/extremes.txt
gs://ssudhir2-stat606s24-hw11/ps_year_avgs.py
gs://ssudhir2-stat606s24-hw11/ps_year_extremes.py
gs://ssudhir2-stat606s24-hw11/wp_output.txt
gs://ssudhir2-stat606s24-hw11/WP_averages/
gs://ssudhir2-stat606s24-hw11/WP_extremes/
gs://ssudhir2-stat606s24-hw11/WP_wordcount/
```

Finally, we check our running clusters, and delete them as required:

```bash
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters list --region=us-east1
NAME: ssudhir2-stat606s24-hw11
PLATFORM: GCE
PRIMARY_WORKER_COUNT: 2
SECONDARY_WORKER_COUNT: 
STATUS: RUNNING
ZONE: us-east1-d
SCHEDULED_DELETE: 
ssudhir2@cloudshell:~ (ssudhir2-stat606s24)$ gcloud dataproc clusters delete ssudhir2-stat606s24-hw11 --region=us-east1
The cluster 'ssudhir2-stat606s24-hw11' and all attached disks will be deleted.

Do you want to continue (Y/n)?  Y

Waiting on operation [projects/ssudhir2-stat606s24/regions/us-east1/operations/68b6ca5d-6422-3dcb-a817-b5b17b8d2d93].
Waiting for cluster deletion operation...done.                                                                                                                             
Deleted [https://dataproc.googleapis.com/v1/projects/ssudhir2-stat606s24/regions/us-east1/clusters/ssudhir2-stat606s24-hw11].
```

## Additional Note:

*I installed `emacs` in the SSH-in-browswer command terminal of my dataproc cluster to allow me to edit texts within the terminal itself. I did this by running the following code:*

```bash
ssudhir2@ssudhir2-stat606s24-hw11-m:~$ sudo apt-get install emacs
Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  adwaita-icon-theme at-spi2-core dbus-user-session dconf-gsettings-backend
  dconf-service emacs-bin-common emacs-common emacs-el emacs-gtk emacsen-common
  ghostscript glib-networking glib-networking-common glib-networking-services
  gsettings-desktop-schemas gsfonts gtk-update-icon-cache hicolor-icon-theme
  imagemagick-6-common install-info libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libcairo-gobject2 libcolord2 libcroco3 libdconf1
  libde265-0 libepoxy0 libfftw3-double3 libgd3 libgdk-pixbuf2.0-0
  libgdk-pixbuf2.0-bin libgdk-pixbuf2.0-common libgif7 libgtk-3-0 libgtk-3-bin
  libgtk-3-common libheif1 libjson-glib-1.0-0 libjson-glib-1.0-common
  liblqr-1-0 libm17n-0 libmagickcore-6.q16-6 libmagickwand-6.q16-6 libotf0
  libproxy1v5 librest-0.7-0 librsvg2-2 librsvg2-common libsoup-gnome2.4-1
  libsoup2.4-1 libwayland-cursor0 libwayland-egl1 libwebpmux3 libx265-165
  libxkbcommon0 m17n-db xkb-data
Suggested packages:
  emacs-common-non-dfsg ncurses-term ghostscript-x colord libfftw3-bin
  libfftw3-dev libgd-tools gvfs m17n-docs libmagickcore-6.q16-6-extra
  librsvg2-bin gawk
The following NEW packages will be installed:
  adwaita-icon-theme at-spi2-core dbus-user-session dconf-gsettings-backend
  dconf-service emacs emacs-bin-common emacs-common emacs-el emacs-gtk
  emacsen-common ghostscript glib-networking glib-networking-common
  glib-networking-services gsettings-desktop-schemas gsfonts
  gtk-update-icon-cache hicolor-icon-theme imagemagick-6-common install-info
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libcairo-gobject2
  libcolord2 libcroco3 libdconf1 libde265-0 libepoxy0 libfftw3-double3 libgd3
  libgdk-pixbuf2.0-0 libgdk-pixbuf2.0-bin libgdk-pixbuf2.0-common libgif7
  libgtk-3-0 libgtk-3-bin libgtk-3-common libheif1 libjson-glib-1.0-0
  libjson-glib-1.0-common liblqr-1-0 libm17n-0 libmagickcore-6.q16-6
  libmagickwand-6.q16-6 libotf0 libproxy1v5 librest-0.7-0 librsvg2-2
  librsvg2-common libsoup-gnome2.4-1 libsoup2.4-1 libwayland-cursor0
  libwayland-egl1 libwebpmux3 libx265-165 libxkbcommon0 m17n-db xkb-data
0 upgraded, 61 newly installed, 0 to remove and 5 not upgraded.
Need to get 66.2 MB of archives.
After this operation, 261 MB of additional disk space will be used.
Do you want to continue? [Y/n] Y
Get:1 https://deb.debian.org/debian buster/main amd64 install-info amd64 6.5.0.dfsg.1-4+b1 [343 kB]
Get:2 https://deb.debian.org/debian buster/main amd64 libfftw3-double3 amd64 3.3.8-2 [733 kB]
Get:3 https://deb.debian.org/debian-security buster/updates/main amd64 libde265-0 amd64 1.0.11-0+deb10u6 [180 kB]
Get:4 https://deb.debian.org/debian buster/main amd64 libx265-165 amd64 2.9-4 [1041 kB]
Get:5 https://deb.debian.org/debian buster/main amd64 libheif1 amd64 1.3.2-2~deb10u1 [127 kB]
Get:6 https://deb.debian.org/debian buster/main amd64 liblqr-1-0 amd64 0.4.2-2.1 [29.1 kB]
Get:7 https://deb.debian.org/debian-security buster/updates/main amd64 libwebpmux3 amd64 0.6.1-2+deb10u3 [98.1 kB]
Get:8 https://deb.debian.org/debian-security buster/updates/main amd64 imagemagick-6-common all 8:6.9.10.23+dfsg-2.1+deb10u7 [203 kB]
Get:9 https://deb.debian.org/debian-security buster/updates/main amd64 libmagickcore-6.q16-6 amd64 8:6.9.10.23+dfsg-2.1+deb10u7 [1793 kB]
Get:10 https://deb.debian.org/debian-security buster/updates/main amd64 libmagickwand-6.q16-6 amd64 8:6.9.10.23+dfsg-2.1+deb10u7 [448 kB]
Get:11 https://deb.debian.org/debian buster/main amd64 hicolor-icon-theme all 0.17-2 [11.4 kB]
Get:12 https://deb.debian.org/debian buster/main amd64 libgdk-pixbuf2.0-common all 2.38.1+dfsg-1 [316 kB]
Get:13 https://deb.debian.org/debian buster/main amd64 libgdk-pixbuf2.0-0 amd64 2.38.1+dfsg-1 [177 kB]
Get:14 https://deb.debian.org/debian buster/main amd64 gtk-update-icon-cache amd64 3.24.5-1 [81.7 kB]
Get:15 https://deb.debian.org/debian buster/main amd64 libcroco3 amd64 0.6.12-3 [145 kB]
Get:16 https://deb.debian.org/debian buster/main amd64 librsvg2-2 amd64 2.44.10-2.1+deb10u3 [1258 kB]
Get:17 https://deb.debian.org/debian buster/main amd64 librsvg2-common amd64 2.44.10-2.1+deb10u3 [23.6 kB]
Get:18 https://deb.debian.org/debian buster/main amd64 adwaita-icon-theme all 3.30.1-1 [11.7 MB]
Get:19 https://deb.debian.org/debian buster/main amd64 libatspi2.0-0 amd64 2.30.0-7 [65.0 kB]
Get:20 https://deb.debian.org/debian buster/main amd64 at-spi2-core amd64 2.30.0-7 [70.7 kB]
Get:21 https://deb.debian.org/debian-security buster/updates/main amd64 dbus-user-session amd64 1.12.28-0+deb10u1 [98.6 kB]
Get:22 https://deb.debian.org/debian buster/main amd64 libdconf1 amd64 0.30.1-2 [40.7 kB]
Get:23 https://deb.debian.org/debian buster/main amd64 dconf-service amd64 0.30.1-2 [36.4 kB]
Get:24 https://deb.debian.org/debian buster/main amd64 dconf-gsettings-backend amd64 0.30.1-2 [28.9 kB]
Get:25 https://deb.debian.org/debian buster/main amd64 emacsen-common all 3.0.4 [19.3 kB]
Get:26 https://deb.debian.org/debian-security buster/updates/main amd64 emacs-common all 1:26.1+1-3.2+deb10u4 [13.4 MB]
Get:27 https://deb.debian.org/debian-security buster/updates/main amd64 emacs-bin-common amd64 1:26.1+1-3.2+deb10u4 [145 kB]
Get:28 https://deb.debian.org/debian buster/main amd64 libatk1.0-data all 2.30.0-2 [145 kB]
Get:29 https://deb.debian.org/debian buster/main amd64 libatk1.0-0 amd64 2.30.0-2 [50.6 kB]
Get:30 https://deb.debian.org/debian buster/main amd64 libcairo-gobject2 amd64 1.16.0-4+deb10u1 [125 kB]
Get:31 https://deb.debian.org/debian-security buster/updates/main amd64 libgif7 amd64 5.1.4-3+deb10u1 [43.4 kB]
Get:32 https://deb.debian.org/debian buster/main amd64 libatk-bridge2.0-0 amd64 2.30.0-5 [61.6 kB]
Get:33 https://deb.debian.org/debian buster/main amd64 libcolord2 amd64 1.4.3-4 [141 kB]
Get:34 https://deb.debian.org/debian buster/main amd64 libepoxy0 amd64 1.5.3-0.1 [190 kB]
Get:35 https://deb.debian.org/debian buster/main amd64 libjson-glib-1.0-common all 1.4.4-2 [52.3 kB]
Get:36 https://deb.debian.org/debian buster/main amd64 libjson-glib-1.0-0 amd64 1.4.4-2 [61.2 kB]
Get:37 https://deb.debian.org/debian buster/main amd64 libproxy1v5 amd64 0.4.15-5+deb10u1 [56.1 kB]
Get:38 https://deb.debian.org/debian buster/main amd64 glib-networking-common all 2.58.0-2+deb10u2 [59.5 kB]
Get:39 https://deb.debian.org/debian buster/main amd64 glib-networking-services amd64 2.58.0-2+deb10u2 [13.7 kB]
Get:40 https://deb.debian.org/debian buster/main amd64 gsettings-desktop-schemas all 3.28.1-1 [529 kB]
Get:41 https://deb.debian.org/debian buster/main amd64 glib-networking amd64 2.58.0-2+deb10u2 [54.6 kB]
Get:42 https://deb.debian.org/debian buster/main amd64 libsoup2.4-1 amd64 2.64.2-2 [253 kB]
Get:43 https://deb.debian.org/debian buster/main amd64 libsoup-gnome2.4-1 amd64 2.64.2-2 [18.0 kB]
Get:44 https://deb.debian.org/debian buster/main amd64 librest-0.7-0 amd64 0.8.1-1 [33.7 kB]
Get:45 https://deb.debian.org/debian buster/main amd64 libwayland-cursor0 amd64 1.16.0-1 [14.1 kB]
Get:46 https://deb.debian.org/debian buster/main amd64 libwayland-egl1 amd64 1.16.0-1 [8204 B]
Get:47 https://deb.debian.org/debian buster/main amd64 xkb-data all 2.26-2 [681 kB]
Get:48 https://deb.debian.org/debian buster/main amd64 libxkbcommon0 amd64 0.8.2-1 [123 kB]
Get:49 https://deb.debian.org/debian buster/main amd64 libgtk-3-common all 3.24.5-1 [3678 kB]
Get:50 https://deb.debian.org/debian buster/main amd64 libgtk-3-0 amd64 3.24.5-1 [2580 kB]
Get:51 https://deb.debian.org/debian buster/main amd64 m17n-db all 1.8.0-1 [1279 kB]
Get:52 https://deb.debian.org/debian-security buster/updates/main amd64 libgd3 amd64 2.2.5-5.2+deb10u1 [137 kB]
Get:53 https://deb.debian.org/debian buster/main amd64 libotf0 amd64 0.9.13-4 [53.5 kB]
Get:54 https://deb.debian.org/debian buster/main amd64 libm17n-0 amd64 1.8.0-2 [255 kB]
Get:55 https://deb.debian.org/debian-security buster/updates/main amd64 emacs-gtk amd64 1:26.1+1-3.2+deb10u4 [3552 kB]
Get:56 https://deb.debian.org/debian-security buster/updates/main amd64 emacs all 1:26.1+1-3.2+deb10u4 [45.0 kB]
Get:57 https://deb.debian.org/debian-security buster/updates/main amd64 emacs-el all 1:26.1+1-3.2+deb10u4 [15.9 MB]
Get:58 https://deb.debian.org/debian-security buster/updates/main amd64 ghostscript amd64 9.27~dfsg-2+deb10u9 [95.5 kB]
Get:59 https://deb.debian.org/debian buster/main amd64 gsfonts all 1:8.11+urwcyr1.0.7~pre44-4.4 [3125 kB]
Get:60 https://deb.debian.org/debian buster/main amd64 libgdk-pixbuf2.0-bin amd64 2.38.1+dfsg-1 [24.1 kB]
Get:61 https://deb.debian.org/debian buster/main amd64 libgtk-3-bin amd64 3.24.5-1 [114 kB]
Fetched 66.2 MB in 1s (76.7 MB/s)        
Extracting templates from packages: 100%
Selecting previously unselected package install-info.
(Reading database ... 174487 files and directories currently installed.)
Preparing to unpack .../install-info_6.5.0.dfsg.1-4+b1_amd64.deb ...
Unpacking install-info (6.5.0.dfsg.1-4+b1) ...
Setting up install-info (6.5.0.dfsg.1-4+b1) ...
Selecting previously unselected package libfftw3-double3:amd64.
(Reading database ... 174502 files and directories currently installed.)
Preparing to unpack .../00-libfftw3-double3_3.3.8-2_amd64.deb ...
Unpacking libfftw3-double3:amd64 (3.3.8-2) ...
Selecting previously unselected package libde265-0:amd64.
Preparing to unpack .../01-libde265-0_1.0.11-0+deb10u6_amd64.deb ...
Unpacking libde265-0:amd64 (1.0.11-0+deb10u6) ...
Selecting previously unselected package libx265-165:amd64.
Preparing to unpack .../02-libx265-165_2.9-4_amd64.deb ...
Unpacking libx265-165:amd64 (2.9-4) ...
Selecting previously unselected package libheif1:amd64.
Preparing to unpack .../03-libheif1_1.3.2-2~deb10u1_amd64.deb ...
Unpacking libheif1:amd64 (1.3.2-2~deb10u1) ...
Selecting previously unselected package liblqr-1-0:amd64.
Preparing to unpack .../04-liblqr-1-0_0.4.2-2.1_amd64.deb ...
Unpacking liblqr-1-0:amd64 (0.4.2-2.1) ...
Selecting previously unselected package libwebpmux3:amd64.
Preparing to unpack .../05-libwebpmux3_0.6.1-2+deb10u3_amd64.deb ...
Unpacking libwebpmux3:amd64 (0.6.1-2+deb10u3) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../06-imagemagick-6-common_8%3a6.9.10.23+dfsg-2.1+deb10u7_all.deb ...
Unpacking imagemagick-6-common (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Selecting previously unselected package libmagickcore-6.q16-6:amd64.
Preparing to unpack .../07-libmagickcore-6.q16-6_8%3a6.9.10.23+dfsg-2.1+deb10u7_amd64.deb ...
Unpacking libmagickcore-6.q16-6:amd64 (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Selecting previously unselected package libmagickwand-6.q16-6:amd64.
Preparing to unpack .../08-libmagickwand-6.q16-6_8%3a6.9.10.23+dfsg-2.1+deb10u7_amd64.deb ...
Unpacking libmagickwand-6.q16-6:amd64 (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Selecting previously unselected package hicolor-icon-theme.
Preparing to unpack .../09-hicolor-icon-theme_0.17-2_all.deb ...
Unpacking hicolor-icon-theme (0.17-2) ...
Selecting previously unselected package libgdk-pixbuf2.0-common.
Preparing to unpack .../10-libgdk-pixbuf2.0-common_2.38.1+dfsg-1_all.deb ...
Unpacking libgdk-pixbuf2.0-common (2.38.1+dfsg-1) ...
Selecting previously unselected package libgdk-pixbuf2.0-0:amd64.
Preparing to unpack .../11-libgdk-pixbuf2.0-0_2.38.1+dfsg-1_amd64.deb ...
Unpacking libgdk-pixbuf2.0-0:amd64 (2.38.1+dfsg-1) ...
Selecting previously unselected package gtk-update-icon-cache.
Preparing to unpack .../12-gtk-update-icon-cache_3.24.5-1_amd64.deb ...
No diversion 'diversion of /usr/sbin/update-icon-caches to /usr/sbin/update-icon-caches.gtk2 by libgtk-3-bin', none removed.
No diversion 'diversion of /usr/share/man/man8/update-icon-caches.8.gz to /usr/share/man/man8/update-icon-caches.gtk2.8.gz by libgtk-3-bin', none removed.
Unpacking gtk-update-icon-cache (3.24.5-1) ...
Selecting previously unselected package libcroco3:amd64.
Preparing to unpack .../13-libcroco3_0.6.12-3_amd64.deb ...
Unpacking libcroco3:amd64 (0.6.12-3) ...
Selecting previously unselected package librsvg2-2:amd64.
Preparing to unpack .../14-librsvg2-2_2.44.10-2.1+deb10u3_amd64.deb ...
Unpacking librsvg2-2:amd64 (2.44.10-2.1+deb10u3) ...
Selecting previously unselected package librsvg2-common:amd64.
Preparing to unpack .../15-librsvg2-common_2.44.10-2.1+deb10u3_amd64.deb ...
Unpacking librsvg2-common:amd64 (2.44.10-2.1+deb10u3) ...
Selecting previously unselected package adwaita-icon-theme.
Preparing to unpack .../16-adwaita-icon-theme_3.30.1-1_all.deb ...
Unpacking adwaita-icon-theme (3.30.1-1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../17-libatspi2.0-0_2.30.0-7_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.30.0-7) ...
Selecting previously unselected package at-spi2-core.
Preparing to unpack .../18-at-spi2-core_2.30.0-7_amd64.deb ...
Unpacking at-spi2-core (2.30.0-7) ...
Selecting previously unselected package dbus-user-session.
Preparing to unpack .../19-dbus-user-session_1.12.28-0+deb10u1_amd64.deb ...
Unpacking dbus-user-session (1.12.28-0+deb10u1) ...
Selecting previously unselected package libdconf1:amd64.
Preparing to unpack .../20-libdconf1_0.30.1-2_amd64.deb ...
Unpacking libdconf1:amd64 (0.30.1-2) ...
Selecting previously unselected package dconf-service.
Preparing to unpack .../21-dconf-service_0.30.1-2_amd64.deb ...
Unpacking dconf-service (0.30.1-2) ...
Selecting previously unselected package dconf-gsettings-backend:amd64.
Preparing to unpack .../22-dconf-gsettings-backend_0.30.1-2_amd64.deb ...
Unpacking dconf-gsettings-backend:amd64 (0.30.1-2) ...
Selecting previously unselected package emacsen-common.
Preparing to unpack .../23-emacsen-common_3.0.4_all.deb ...
Unpacking emacsen-common (3.0.4) ...
Selecting previously unselected package emacs-common.
Preparing to unpack .../24-emacs-common_1%3a26.1+1-3.2+deb10u4_all.deb ...
Unpacking emacs-common (1:26.1+1-3.2+deb10u4) ...
Selecting previously unselected package emacs-bin-common.
Preparing to unpack .../25-emacs-bin-common_1%3a26.1+1-3.2+deb10u4_amd64.deb ...
Unpacking emacs-bin-common (1:26.1+1-3.2+deb10u4) ...
Selecting previously unselected package libatk1.0-data.
Preparing to unpack .../26-libatk1.0-data_2.30.0-2_all.deb ...
Unpacking libatk1.0-data (2.30.0-2) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../27-libatk1.0-0_2.30.0-2_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.30.0-2) ...
Selecting previously unselected package libcairo-gobject2:amd64.
Preparing to unpack .../28-libcairo-gobject2_1.16.0-4+deb10u1_amd64.deb ...
Unpacking libcairo-gobject2:amd64 (1.16.0-4+deb10u1) ...
Selecting previously unselected package libgif7:amd64.
Preparing to unpack .../29-libgif7_5.1.4-3+deb10u1_amd64.deb ...
Unpacking libgif7:amd64 (5.1.4-3+deb10u1) ...
Selecting previously unselected package libatk-bridge2.0-0:amd64.
Preparing to unpack .../30-libatk-bridge2.0-0_2.30.0-5_amd64.deb ...
Unpacking libatk-bridge2.0-0:amd64 (2.30.0-5) ...
Selecting previously unselected package libcolord2:amd64.
Preparing to unpack .../31-libcolord2_1.4.3-4_amd64.deb ...
Unpacking libcolord2:amd64 (1.4.3-4) ...
Selecting previously unselected package libepoxy0:amd64.
Preparing to unpack .../32-libepoxy0_1.5.3-0.1_amd64.deb ...
Unpacking libepoxy0:amd64 (1.5.3-0.1) ...
Selecting previously unselected package libjson-glib-1.0-common.
Preparing to unpack .../33-libjson-glib-1.0-common_1.4.4-2_all.deb ...
Unpacking libjson-glib-1.0-common (1.4.4-2) ...
Selecting previously unselected package libjson-glib-1.0-0:amd64.
Preparing to unpack .../34-libjson-glib-1.0-0_1.4.4-2_amd64.deb ...
Unpacking libjson-glib-1.0-0:amd64 (1.4.4-2) ...
Selecting previously unselected package libproxy1v5:amd64.
Preparing to unpack .../35-libproxy1v5_0.4.15-5+deb10u1_amd64.deb ...
Unpacking libproxy1v5:amd64 (0.4.15-5+deb10u1) ...
Selecting previously unselected package glib-networking-common.
Preparing to unpack .../36-glib-networking-common_2.58.0-2+deb10u2_all.deb ...
Unpacking glib-networking-common (2.58.0-2+deb10u2) ...
Selecting previously unselected package glib-networking-services.
Preparing to unpack .../37-glib-networking-services_2.58.0-2+deb10u2_amd64.deb ...
Unpacking glib-networking-services (2.58.0-2+deb10u2) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../38-gsettings-desktop-schemas_3.28.1-1_all.deb ...
Unpacking gsettings-desktop-schemas (3.28.1-1) ...
Selecting previously unselected package glib-networking:amd64.
Preparing to unpack .../39-glib-networking_2.58.0-2+deb10u2_amd64.deb ...
Unpacking glib-networking:amd64 (2.58.0-2+deb10u2) ...
Selecting previously unselected package libsoup2.4-1:amd64.
Preparing to unpack .../40-libsoup2.4-1_2.64.2-2_amd64.deb ...
Unpacking libsoup2.4-1:amd64 (2.64.2-2) ...
Selecting previously unselected package libsoup-gnome2.4-1:amd64.
Preparing to unpack .../41-libsoup-gnome2.4-1_2.64.2-2_amd64.deb ...
Unpacking libsoup-gnome2.4-1:amd64 (2.64.2-2) ...
Selecting previously unselected package librest-0.7-0:amd64.
Preparing to unpack .../42-librest-0.7-0_0.8.1-1_amd64.deb ...
Unpacking librest-0.7-0:amd64 (0.8.1-1) ...
Selecting previously unselected package libwayland-cursor0:amd64.
Preparing to unpack .../43-libwayland-cursor0_1.16.0-1_amd64.deb ...
Unpacking libwayland-cursor0:amd64 (1.16.0-1) ...
Selecting previously unselected package libwayland-egl1:amd64.
Preparing to unpack .../44-libwayland-egl1_1.16.0-1_amd64.deb ...
Unpacking libwayland-egl1:amd64 (1.16.0-1) ...
Selecting previously unselected package xkb-data.
Preparing to unpack .../45-xkb-data_2.26-2_all.deb ...
Unpacking xkb-data (2.26-2) ...
Selecting previously unselected package libxkbcommon0:amd64.
Preparing to unpack .../46-libxkbcommon0_0.8.2-1_amd64.deb ...
Unpacking libxkbcommon0:amd64 (0.8.2-1) ...
Selecting previously unselected package libgtk-3-common.
Preparing to unpack .../47-libgtk-3-common_3.24.5-1_all.deb ...
Unpacking libgtk-3-common (3.24.5-1) ...
Selecting previously unselected package libgtk-3-0:amd64.
Preparing to unpack .../48-libgtk-3-0_3.24.5-1_amd64.deb ...
Unpacking libgtk-3-0:amd64 (3.24.5-1) ...
Selecting previously unselected package m17n-db.
Preparing to unpack .../49-m17n-db_1.8.0-1_all.deb ...
Unpacking m17n-db (1.8.0-1) ...
Selecting previously unselected package libgd3:amd64.
Preparing to unpack .../50-libgd3_2.2.5-5.2+deb10u1_amd64.deb ...
Unpacking libgd3:amd64 (2.2.5-5.2+deb10u1) ...
Selecting previously unselected package libotf0:amd64.
Preparing to unpack .../51-libotf0_0.9.13-4_amd64.deb ...
Unpacking libotf0:amd64 (0.9.13-4) ...
Selecting previously unselected package libm17n-0:amd64.
Preparing to unpack .../52-libm17n-0_1.8.0-2_amd64.deb ...
Unpacking libm17n-0:amd64 (1.8.0-2) ...
Selecting previously unselected package emacs-gtk.
Preparing to unpack .../53-emacs-gtk_1%3a26.1+1-3.2+deb10u4_amd64.deb ...
Unpacking emacs-gtk (1:26.1+1-3.2+deb10u4) ...
Selecting previously unselected package emacs.
Preparing to unpack .../54-emacs_1%3a26.1+1-3.2+deb10u4_all.deb ...
Unpacking emacs (1:26.1+1-3.2+deb10u4) ...
Selecting previously unselected package emacs-el.
Preparing to unpack .../55-emacs-el_1%3a26.1+1-3.2+deb10u4_all.deb ...
Unpacking emacs-el (1:26.1+1-3.2+deb10u4) ...
Selecting previously unselected package ghostscript.
Preparing to unpack .../56-ghostscript_9.27~dfsg-2+deb10u9_amd64.deb ...
Unpacking ghostscript (9.27~dfsg-2+deb10u9) ...
Selecting previously unselected package gsfonts.
Preparing to unpack .../57-gsfonts_1%3a8.11+urwcyr1.0.7~pre44-4.4_all.deb ...
Unpacking gsfonts (1:8.11+urwcyr1.0.7~pre44-4.4) ...
Selecting previously unselected package libgdk-pixbuf2.0-bin.
Preparing to unpack .../58-libgdk-pixbuf2.0-bin_2.38.1+dfsg-1_amd64.deb ...
Unpacking libgdk-pixbuf2.0-bin (2.38.1+dfsg-1) ...
Selecting previously unselected package libgtk-3-bin.
Preparing to unpack .../59-libgtk-3-bin_3.24.5-1_amd64.deb ...
Unpacking libgtk-3-bin (3.24.5-1) ...
Setting up libotf0:amd64 (0.9.13-4) ...
Setting up imagemagick-6-common (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Setting up libproxy1v5:amd64 (0.4.15-5+deb10u1) ...
Setting up hicolor-icon-theme (0.17-2) ...
Setting up libx265-165:amd64 (2.9-4) ...
Setting up libgdk-pixbuf2.0-common (2.38.1+dfsg-1) ...
Setting up m17n-db (1.8.0-1) ...
Setting up xkb-data (2.26-2) ...
Setting up libatspi2.0-0:amd64 (2.30.0-7) ...
Setting up ghostscript (9.27~dfsg-2+deb10u9) ...
Setting up libcolord2:amd64 (1.4.3-4) ...
Setting up libdconf1:amd64 (0.30.1-2) ...
Setting up dbus-user-session (1.12.28-0+deb10u1) ...
Setting up emacsen-common (3.0.4) ...
Setting up libepoxy0:amd64 (1.5.3-0.1) ...
Setting up gsfonts (1:8.11+urwcyr1.0.7~pre44-4.4) ...
Setting up libgdk-pixbuf2.0-0:amd64 (2.38.1+dfsg-1) ...
Setting up libgd3:amd64 (2.2.5-5.2+deb10u1) ...
Setting up libcroco3:amd64 (0.6.12-3) ...
Setting up libfftw3-double3:amd64 (3.3.8-2) ...
Setting up libgif7:amd64 (5.1.4-3+deb10u1) ...
Setting up libatk1.0-data (2.30.0-2) ...
Setting up liblqr-1-0:amd64 (0.4.2-2.1) ...
Setting up at-spi2-core (2.30.0-7) ...
Setting up libgdk-pixbuf2.0-bin (2.38.1+dfsg-1) ...
Setting up libwayland-cursor0:amd64 (1.16.0-1) ...
Setting up libjson-glib-1.0-common (1.4.4-2) ...
Setting up libcairo-gobject2:amd64 (1.16.0-4+deb10u1) ...
Setting up libatk1.0-0:amd64 (2.30.0-2) ...
Setting up libwayland-egl1:amd64 (1.16.0-1) ...
Setting up glib-networking-common (2.58.0-2+deb10u2) ...
Setting up emacs-common (1:26.1+1-3.2+deb10u4) ...
Setting up libde265-0:amd64 (1.0.11-0+deb10u6) ...
Setting up libwebpmux3:amd64 (0.6.1-2+deb10u3) ...
Setting up libxkbcommon0:amd64 (0.8.2-1) ...
Setting up glib-networking-services (2.58.0-2+deb10u2) ...
Setting up gtk-update-icon-cache (3.24.5-1) ...
Setting up libheif1:amd64 (1.3.2-2~deb10u1) ...
Setting up dconf-service (0.30.1-2) ...
Setting up libm17n-0:amd64 (1.8.0-2) ...
Setting up libjson-glib-1.0-0:amd64 (1.4.4-2) ...
Setting up librsvg2-2:amd64 (2.44.10-2.1+deb10u3) ...
Setting up libatk-bridge2.0-0:amd64 (2.30.0-5) ...
Setting up libmagickcore-6.q16-6:amd64 (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Setting up emacs-el (1:26.1+1-3.2+deb10u4) ...
Setting up librsvg2-common:amd64 (2.44.10-2.1+deb10u3) ...
Setting up emacs-bin-common (1:26.1+1-3.2+deb10u4) ...
update-alternatives: using /usr/bin/ctags.emacs to provide /usr/bin/ctags (ctags) in auto mode
update-alternatives: using /usr/bin/ebrowse.emacs to provide /usr/bin/ebrowse (ebrowse) in auto mode
update-alternatives: using /usr/bin/emacsclient.emacs to provide /usr/bin/emacsclient (emacsclient) in auto mode
update-alternatives: using /usr/bin/etags.emacs to provide /usr/bin/etags (etags) in auto mode
Setting up libmagickwand-6.q16-6:amd64 (8:6.9.10.23+dfsg-2.1+deb10u7) ...
Setting up dconf-gsettings-backend:amd64 (0.30.1-2) ...
Setting up adwaita-icon-theme (3.30.1-1) ...
update-alternatives: using /usr/share/icons/Adwaita/cursor.theme to provide /usr/share/icons/default/index.theme (x-cursor-theme) in auto mode
Setting up libgtk-3-common (3.24.5-1) ...
Setting up gsettings-desktop-schemas (3.28.1-1) ...
Processing triggers for mime-support (3.62) ...
Processing triggers for libglib2.0-0:amd64 (2.58.3-2+deb10u5) ...
Processing triggers for libc-bin (2.28-10+deb10u2) ...
Processing triggers for man-db (2.8.5-2+deb10u1) ...
Setting up glib-networking:amd64 (2.58.0-2+deb10u2) ...
Processing triggers for install-info (6.5.0.dfsg.1-4+b1) ...
Processing triggers for fontconfig (2.13.1-2) ...
Setting up libsoup2.4-1:amd64 (2.64.2-2) ...
Setting up libsoup-gnome2.4-1:amd64 (2.64.2-2) ...
Setting up librest-0.7-0:amd64 (0.8.1-1) ...
Setting up libgtk-3-0:amd64 (3.24.5-1) ...
Setting up libgtk-3-bin (3.24.5-1) ...
Setting up emacs-gtk (1:26.1+1-3.2+deb10u4) ...
update-alternatives: using /usr/bin/emacs-gtk to provide /usr/bin/emacs (emacs) in auto mode
Install emacsen-common for emacs
emacsen-common: Handling install of emacsen flavor emacs
Setting up emacs (1:26.1+1-3.2+deb10u4) ...
Processing triggers for libgdk-pixbuf2.0-0:amd64 (2.38.1+dfsg-1) ...
Processing triggers for libc-bin (2.28-10+deb10u2) ...
```